# 🤖 02 — NLP Classique : BoW, TF-IDF & Modèles Classiques

**Objectif :** Construire, entraîner et comparer les modèles de la Phase 1.

**Pipeline :**
```
Texte brut
    ↓  preprocessor.py
Texte nettoyé
    ↓  CountVectorizer / TfidfVectorizer
Matrice numérique (sparse)
    ↓  Naive Bayes / Logistic Regression / SVM / Random Forest
Prédiction : Ham (0) ou Spam (1)
```

**Ce notebook importe les modules `src/` pour :**
- Éviter la duplication de code
- Garantir que notebook et script `main.py` utilisent exactement la même logique

## 0. Setup

In [ ]:
import os
import sys
import warnings
warnings.filterwarnings('ignore')

# Ajouter src/ au path pour importer nos modules
# os.path.abspath('..') → répertoire parent du notebook (racine du projet)
project_root = os.path.abspath('..')
sys.path.insert(0, os.path.join(project_root, 'src'))

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('inline')  # Graphiques inline dans Jupyter
import matplotlib.pyplot as plt
import seaborn as sns

from dotenv import load_dotenv
load_dotenv(os.path.join(project_root, '.env'))

# Import de nos modules
from preprocessor import TextPreprocessor
from models import NLPTrainer
from evaluate import ModelEvaluator

sns.set_theme(style='whitegrid', palette='husl')
plt.rcParams['figure.dpi'] = 120
plt.rcParams['figure.facecolor'] = 'white'

# Configuration
DATA_PATH    = os.path.join(project_root, os.getenv('DATA_PATH', 'data/sms_spam.csv'))
MODELS_DIR   = os.path.join(project_root, os.getenv('MODELS_DIR', 'models'))
REPORTS_DIR  = os.path.join(project_root, 'reports')
TEST_SIZE    = float(os.getenv('TEST_SIZE', '0.2'))
RANDOM_STATE = int(os.getenv('RANDOM_STATE', '42'))
LANGUAGE     = os.getenv('LANGUAGE', 'english')

os.makedirs(REPORTS_DIR, exist_ok=True)
os.makedirs(MODELS_DIR,  exist_ok=True)

print('✅ Setup OK')
print(f'   Dataset    : {DATA_PATH}')
print(f'   Test size  : {TEST_SIZE*100:.0f}%')
print(f'   Seed       : {RANDOM_STATE}')

## 1. Chargement et prétraitement

In [ ]:
# ── Chargement ────────────────────────────────────────────────────────────────
df = pd.read_csv(
    DATA_PATH, sep='\t', header=None,
    names=['label', 'text'], encoding='latin-1'
)
df['label'] = df['label'].map({'ham': 0, 'spam': 1})
df = df.dropna().reset_index(drop=True)

print(f'Messages chargés : {len(df)}')
print(f"Distribution : Ham={df['label'].eq(0).sum()}, Spam={df['label'].eq(1).sum()}")
df.head()

In [ ]:
# ── Prétraitement ─────────────────────────────────────────────────────────────
preprocessor = TextPreprocessor(language=LANGUAGE)
df['text_clean'] = preprocessor.clean_series(df['text'])

# Visualisation avant/après
print('AVANT → APRÈS prétraitement\n')
for idx in df[df['label']==1].index[:3]:
    print(f'[SPAM]')
    print(f'  Avant  : {df.loc[idx, "text"][:100]}')
    print(f'  Après  : {df.loc[idx, "text_clean"][:100]}')
    print()

In [ ]:
# ── Impact du prétraitement sur la longueur ───────────────────────────────────
df['len_before'] = df['text'].str.split().str.len()
df['len_after']  = df['text_clean'].str.split().str.len()
df['reduction']  = ((df['len_before'] - df['len_after']) / df['len_before'] * 100).round(1)

print('Impact du prétraitement (nombre de mots) :')
print(df.groupby('label')[['len_before', 'len_after', 'reduction']].mean().round(1))
print('\n→ Le prétraitement réduit le texte en supprimant les mots non-informatifs')

## 2. Compréhension de la vectorisation

**Avant de lancer les modèles, visualisons ce que font BoW et TF-IDF.**

Les algorithmes ML ne comprennent pas les mots — ils comprennent les nombres.
La vectorisation transforme du texte en matrices numériques.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

# Exemple jouet pour comprendre la vectorisation
corpus_exemple = [
    "free prize win now",         # spam-like
    "free call win prize claim",  # spam-like
    "going cinema tonight fun",   # ham-like
]

print('=' * 60)
print('DÉMONSTRATION : BAG OF WORDS')
print('=' * 60)

bow = CountVectorizer()
X_bow = bow.fit_transform(corpus_exemple)

print(f'\nVocabulaire appris ({len(bow.vocabulary_)} mots) :')
# tri par index pour avoir l'ordre des colonnes
vocab_sorted = sorted(bow.vocabulary_.items(), key=lambda x: x[1])
print('  ' + ' | '.join([f'{w:>8}' for w, _ in vocab_sorted]))

print('\nMatrice BoW :')
df_bow = pd.DataFrame(
    X_bow.toarray(),
    columns=[w for w, _ in vocab_sorted],
    index=['Doc 1 (spam)', 'Doc 2 (spam)', 'Doc 3 (ham)']
)
print(df_bow.to_string())

print('\n💡 BoW : compte les occurrences brutes de chaque mot.')
print('   Chaque colonne = un mot du vocabulaire.')
print('   Chaque ligne = un document représenté par ses comptages.')

In [ ]:
print('=' * 60)
print('DÉMONSTRATION : TF-IDF')
print('=' * 60)

tfidf = TfidfVectorizer()
X_tfidf = tfidf.fit_transform(corpus_exemple)

vocab_sorted_tfidf = sorted(tfidf.vocabulary_.items(), key=lambda x: x[1])

df_tfidf = pd.DataFrame(
    X_tfidf.toarray().round(3),
    columns=[w for w, _ in vocab_sorted_tfidf],
    index=['Doc 1 (spam)', 'Doc 2 (spam)', 'Doc 3 (ham)']
)
print('\nMatrice TF-IDF :')
print(df_tfidf.to_string())

print("""
💡 TF-IDF : pondère les mots par leur importance relative.

   TF  (Term Frequency)          = fréquence du mot dans CE document
   IDF (Inverse Doc Frequency)   = rareté du mot dans TOUS les documents
   TF-IDF                        = TF × IDF

   Interprétation :
   - 'free' apparaît dans 2/3 documents → IDF faible → score modéré
   - 'cinema' n'apparaît que dans 1 doc  → IDF élevé  → score fort
   - Un mot présent dans TOUS les docs aurait IDF ≈ 0

   → TF-IDF pénalise les mots ubiquitaires ('the', 'is') et
     récompense les mots spécifiques à certains documents.
""")

In [ ]:
# ── Visualisation comparée BoW vs TF-IDF ──────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

for ax, matrix, title, cmap in zip(
    axes,
    [df_bow, df_tfidf],
    ['Bag of Words (comptages bruts)', 'TF-IDF (scores pondérés)'],
    ['YlOrRd', 'YlGnBu']
):
    sns.heatmap(
        matrix.astype(float),
        annot=True, fmt='.2f', cmap=cmap,
        ax=ax, linewidths=0.5, linecolor='white'
    )
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_xlabel('Mots du vocabulaire')

plt.suptitle('BoW vs TF-IDF — Exemple jouet (3 documents)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{REPORTS_DIR}/vectorization_comparison_example.png', bbox_inches='tight', dpi=150)
plt.show()

## 3. Entraînement des modèles

On utilise `NLPTrainer` qui entraîne les 8 combinaisons (2 vectoriseurs × 4 modèles) et mesure les performances.

In [ ]:
# ── Initialisation ────────────────────────────────────────────────────────────
trainer = NLPTrainer(random_state=RANDOM_STATE, test_size=TEST_SIZE)

X = df['text_clean']
y = df['label']

X_train, X_test, y_train, y_test = trainer.split_data(X, y)

print(f'Taille du train set : {len(X_train)} exemples')
print(f'Taille du test set  : {len(X_test)} exemples')
print(f'Spam dans le test   : {y_test.sum()} ({y_test.mean()*100:.1f}%)')

In [ ]:
# ── Entraînement de tous les modèles ─────────────────────────────────────────
# 2 vectoriseurs × 4 modèles = 8 pipelines
results = trainer.train_all(X_train, X_test, y_train, y_test)

## 4. Comparaison des performances

In [ ]:
# ── Tableau comparatif ────────────────────────────────────────────────────────
df_results = trainer.get_comparison_dataframe()

# Affichage avec mise en forme
print('TABLEAU COMPARATIF DES MODÈLES — PHASE 1')
print('=' * 80)

# highlight_max colore en vert la meilleure valeur de chaque colonne numérique
styled = df_results.style.highlight_max(
    subset=['accuracy', 'precision', 'recall', 'f1_score', 'cv_f1_mean'],
    color='lightgreen'
).highlight_min(
    subset=['train_time_s'],
    color='lightblue'
).format({
    'accuracy'   : '{:.4f}',
    'precision'  : '{:.4f}',
    'recall'     : '{:.4f}',
    'f1_score'   : '{:.4f}',
    'cv_f1_mean' : '{:.4f}',
    'cv_f1_std'  : '{:.4f}',
    'train_time_s': '{:.3f}s',
})

styled

In [ ]:
# ── Graphique comparatif des métriques ───────────────────────────────────────
evaluator = ModelEvaluator(output_dir=REPORTS_DIR)
evaluator.plot_metrics_comparison(results, save=True)
plt.show()

In [ ]:
# ── Graphique temps d'entraînement ───────────────────────────────────────────
evaluator.plot_training_time(results, save=True)
plt.show()

In [ ]:
# ── Matrices de confusion ─────────────────────────────────────────────────────
# Affiche les matrices de confusion pour les modèles TF-IDF (généralement meilleurs)
tfidf_results = [r for r in results if r['vectorizer'] == 'tfidf']

fig, axes = plt.subplots(1, len(tfidf_results), figsize=(16, 4))

for ax, result in zip(axes, tfidf_results):
    import seaborn as sns
    cm = result['confusion_matrix']
    sns.heatmap(
        cm, annot=True, fmt='d', cmap='Blues', ax=ax,
        xticklabels=['Ham', 'Spam'],
        yticklabels=['Ham', 'Spam'],
    )
    ax.set_title(
        f"{result['model']}\nF1={result['f1_score']:.4f}",
        fontweight='bold'
    )
    ax.set_xlabel('Prédit')
    ax.set_ylabel('Réel')

plt.suptitle('Matrices de confusion — TF-IDF', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{REPORTS_DIR}/confusion_matrices_tfidf.png', bbox_inches='tight', dpi=150)
plt.show()

## 5. Analyse détaillée du meilleur modèle

In [ ]:
# ── Rapport de classification complet ────────────────────────────────────────
best_result = max(results, key=lambda r: r['f1_score'])
evaluator.print_classification_report(best_result, y_test)

print(f"""
🏆 Meilleur modèle : {best_result['vectorizer']} + {best_result['model']}

Interprétation du rapport de classification :

  Precision (Ham)  : Sur tous les messages classés 'Ham',
                     quelle fraction était vraiment Ham ?
  
  Recall (Spam)    : Sur tous les vrais Spam,
                     quelle fraction a été détectée ?
                     ← Métrique la plus critique pour un filtre anti-spam
  
  F1-Score (Spam)  : Équilibre Precision/Recall pour la classe Spam.
""")

In [ ]:
# ── Mots les plus influents (Logistic Regression uniquement) ─────────────────
# La Logistic Regression est le seul modèle dont on peut facilement
# extraire les coefficients pour comprendre quelle feature (mot) pèse le plus.

lr_tfidf = next(
    (r for r in results if r['model'] == 'logistic_regression' and r['vectorizer'] == 'tfidf'),
    None
)

if lr_tfidf:
    pipeline  = lr_tfidf['pipeline']
    vectorizer = pipeline.named_steps['vectorizer']
    classifier = pipeline.named_steps['classifier']
    
    # Récupère les noms des features (mots du vocabulaire)
    feature_names = vectorizer.get_feature_names_out()
    
    # Coefficients positifs → prédisent SPAM
    # Coefficients négatifs → prédisent HAM
    coefs = classifier.coef_[0]
    
    top_spam = pd.Series(coefs, index=feature_names).nlargest(20)
    top_ham  = pd.Series(coefs, index=feature_names).nsmallest(20)
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    
    top_spam.plot(kind='barh', ax=axes[0], color='#F44336', alpha=0.85)
    axes[0].set_title('Top 20 mots → SPAM', fontsize=12, fontweight='bold')
    axes[0].set_xlabel('Coefficient (positif = spam)')
    
    top_ham.abs().sort_values().plot(kind='barh', ax=axes[1], color='#4CAF50', alpha=0.85)
    axes[1].set_title('Top 20 mots → HAM', fontsize=12, fontweight='bold')
    axes[1].set_xlabel('|Coefficient| (négatif = ham)')
    
    plt.suptitle('Mots les plus influents — TF-IDF + Logistic Regression',
                 fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig(f'{REPORTS_DIR}/feature_importance_lr_tfidf.png', bbox_inches='tight', dpi=150)
    plt.show()
    
    print('\n💡 Ces mots ont le plus fort impact sur la classification.')
    print('   Un modèle interprétable permet de vérifier que la logique apprise est cohérente.')

## 6. Sauvegarde et test du meilleur modèle

In [ ]:
# ── Sauvegarde ────────────────────────────────────────────────────────────────
filepath = trainer.save_best_model(models_dir=MODELS_DIR)
print(f'✅ Modèle sauvegardé : {filepath}')

In [ ]:
# ── Test du modèle rechargé ───────────────────────────────────────────────────
import joblib

# Rechargement depuis le disque
pipeline = joblib.load(filepath)

# Messages de test
test_messages = [
    "FREE entry! Win a £1000 prize! Call 09061743853 NOW!",   # spam évident
    "Hey, are you coming to the party tonight?",               # ham évident
    "Congratulations! You've been selected for a cash prize!", # spam
    "Can you pick up some milk on your way home?",             # ham
    "URGENT: Your account will be suspended. Click here NOW",  # spam
]

# Prétraitement avant prédiction
preprocessor = TextPreprocessor(language=LANGUAGE)
test_clean = [preprocessor.clean(msg) for msg in test_messages]

predictions = pipeline.predict(test_clean)
labels = {0: '✅ HAM ', 1: '🚨 SPAM'}

print('TEST DU MODÈLE SUR DES NOUVEAUX MESSAGES')
print('=' * 70)
for msg, pred in zip(test_messages, predictions):
    print(f'{labels[pred]} | {msg[:65]}')

print(f'\n✅ Pipeline rechargé et fonctionnel : {os.path.basename(filepath)}')

## 7. Tableau comparatif Phase 1 — Synthèse

Ce tableau sera enrichi à chaque nouvelle phase du projet.

In [ ]:
import pandas as pd

# Résumé des méthodes de vectorisation comparées
comparison = pd.DataFrame([
    {
        'Méthode'         : 'Bag of Words',
        'Principe'        : 'Comptage brut des occurrences',
        'Avantages'       : 'Simple, rapide, interprétable',
        'Inconvénients'   : 'Ignore la fréquence relative, ordre des mots',
        'Mémoire'         : 'Faible',
        'Vitesse'         : 'Très rapide',
        'Recommandé pour' : 'Baseline, petits datasets',
    },
    {
        'Méthode'         : 'TF-IDF',
        'Principe'        : 'Pondération par importance relative',
        'Avantages'       : 'Meilleur que BoW, réduit le bruit',
        'Inconvénients'   : 'Ignore toujours le sens et l\'ordre',
        'Mémoire'         : 'Faible',
        'Vitesse'         : 'Très rapide',
        'Recommandé pour' : 'Classification texte, recherche d\'information',
    },
    {
        'Méthode'         : 'Word2Vec (Phase 2)',
        'Principe'        : 'Embeddings denses (vecteurs 300D)',
        'Avantages'       : 'Capture la sémantique (roi-homme+femme=reine)',
        'Inconvénients'   : 'Besoin de beaucoup de données, moins interprétable',
        'Mémoire'         : 'Moyen',
        'Vitesse'         : 'Moyen',
        'Recommandé pour' : 'NLP avancé, similarité sémantique',
    },
    {
        'Méthode'         : 'BERT (Phase 3)',
        'Principe'        : 'Transformer pré-entraîné (contexte bidirectionnel)',
        'Avantages'       : 'State-of-the-art, comprend le contexte',
        'Inconvénients'   : 'Lent, gourmand en mémoire, GPU requis',
        'Mémoire'         : 'Élevée (GPU)',
        'Vitesse'         : 'Lent',
        'Recommandé pour' : 'Haute précision requise, ressources disponibles',
    },
])

print('TABLEAU COMPARATIF DES MÉTHODES NLP — TOUTES PHASES')
print('(Les phases 2 et 3 seront complétées progressivement)')
print()

# Affichage formaté
pd.set_option('display.max_colwidth', 50)
comparison.style.set_properties(**{'text-align': 'left'}).set_table_styles(
    [{'selector': 'th', 'props': [('font-weight', 'bold'), ('text-align', 'left')]}]
)

In [ ]:
# Résumé des performances Phase 1
print('PERFORMANCES PHASE 1 — RÉSUMÉ FINAL')
print('=' * 70)
print(df_results.to_string(index=False))

best = df_results.iloc[0]
print(f"""
🏆 Meilleur modèle : {best['vectorizer']} + {best['model']}
   F1-Score  : {best['f1_score']:.4f}
   Precision : {best['precision']:.4f}
   Recall    : {best['recall']:.4f}
   Accuracy  : {best['accuracy']:.4f}
   Temps     : {best['train_time_s']:.3f}s

💡 Prochaine étape (Phase 2) :
   Ajouter Word2Vec et FastText pour capturer
   la sémantique des mots et améliorer ces scores.
""")